In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [ ]:
X_train = pd.read_csv("../data/X_train.csv")
X_test = pd.read_csv("../data/X_test.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze() # random forest expects a 1d array
y_test = pd.read_csv("../data/y_test.csv").squeeze()

# over n iterations, take a unique 1/n sample fold for validation at each iteration, take averge of all validations
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) 

print(f"Train size: {X_train.shape[0]}")
print(f"Test size:  {X_test.shape[0]}")
print(f"\nClass balance in train:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"\nClass balance in test:\n{y_test.value_counts(normalize=True).round(3)}")

In [ ]:
pipeline = {
    "Logistic Regression" : Pipeline([
        ("scalar", StandardScaler()), # standardises features using z = (x - mean)/std to centre data around mean 0
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
    ]),
    "Random Forest" : Pipeline([
        ("model", RandomForestClassifier(class_weight="balanced", n_estimators=100, random_state=42))
    ]),
    "XGBoost" : Pipeline([
        ("model", XGBClassifier(
            scale_pos_weight = (y_train == 0).sum()/(y_train == 1).sum(), # class imbalance, weight ~~ 3, pensalises mistakes on minority class churn
            n_estimators = 100,
            eval_metric = "auc", # auc-roc 
            random_state = 42,
        ))
    ])
}

metrics = ["f1", "roc_auc", "precision", "recall"]

for algorithm, model_pipeline in pipeline.items():
    print(f"\n--{algorithm}")

    cross_validation_results = cross_validate(model_pipeline, X_train, y_train, cv=skf, scoring=metrics)

    for metric in metrics:
        scores = cross_validation_results[f"test_{metric}"]
        print(f"{metric.upper()} per fold: {np.round(scores, 3)}")
        print(f"Mean: {scores.mean():.3f}")
        print(f"Standard Deviation: {scores.std():.3f}\n")

    y_prediction_cross_validation = cross_val_predict(model_pipeline, X_train, y_train, cv=skf)

    cm = confusion_matrix(y_train, y_prediction_cross_validation)
    display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Retained", "Churned"])
    display.plot(cmap="coolwarm")
    plt.title(algorithm)
    plt.tight_layout()
    plt.show()

    print("\nClassification Report:")
    print(classification_report(y_train, y_prediction_cross_validation, target_names=["Retained", "Churned"]))